In [18]:
import pandas as pd
from pathlib import Path
DATA_DIR = Path(r"../datasets/")
PROCESSED_DIR = DATA_DIR / "processed"

movies = pd.read_csv(
    PROCESSED_DIR / "movies_phase1_final.csv"
)

# print(movies.shape)
# print(movies.columns.tolist())

# print("Null content:",
#       movies["content_text"].isna().sum())

# print("Empty content:",
#       movies["content_text"].eq("").sum())

# print("\nSample content:")
# print(
#     movies[
#         ["movieId", "clean_title", "content_text"]
#     ].head(5)
# )

In [ ]:
movies["content_text"] = (
    movies["content_text"]
    .fillna("")
    .astype(str)
    .str.strip()
)
print(movies["content_text"].head())

In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    stop_words="english",
    max_features=20000,
    ngram_range=(1, 2),
    min_df=2
)
tfidf_matrix = tfidf.fit_transform(
    movies["content_text"]
)
# print("TF-IDF matrix shape:", tfidf_matrix.shape)

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
toy_index = movies[
    movies["clean_title"].str.lower() == "toy story"
].index[0]

print("Toy Story index:", toy_index)

similarity_scores = cosine_similarity(
    tfidf_matrix[toy_index],
    tfidf_matrix
).flatten()

similar_indices = np.argsort(similarity_scores)[::-1]
top_indices = similar_indices[1:11]

baseline_result = movies.iloc[top_indices][
    ["movieId", "clean_title", "genres"]
].copy()

baseline_result["similarity"] = similarity_scores[top_indices]

display(baseline_result)


In [14]:
# ============================================================
# CLEAN TEXT COLUMNS BEFORE TF-IDF
# ============================================================

text_columns = [
    "genre_text",
    "tag_text",
    "genome_tag_text",
    "content_text"
]

for column in text_columns:
    movies[column] = (
        movies[column]
        .fillna("")
        .astype(str)
        .str.strip()
    )

# Verify there are no NaN values
print(movies[text_columns].isna().sum())

genre_text         0
tag_text           0
genome_tag_text    0
content_text       0
dtype: int64


In [ ]:

# WEIGHTED TF-IDF FEATURES
from scipy.sparse import hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
genre_tfidf = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=2
)

tag_tfidf = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    min_df=2,
    max_features=10000
)

genome_tfidf = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    min_df=2,
    max_features=10000
)

genre_matrix = genre_tfidf.fit_transform(
    movies["genre_text"]
)

tag_matrix = tag_tfidf.fit_transform(
    movies["tag_text"]
)

genome_matrix = genome_tfidf.fit_transform(
    movies["genome_tag_text"]
)

genre_weight = 3.0
tag_weight = 2.0
genome_weight = 0.5

weighted_matrix = hstack([
    genre_matrix * genre_weight,
    tag_matrix * tag_weight,
    genome_matrix * genome_weight
]).tocsr()

print("Genre matrix:", genre_matrix.shape)
print("Tag matrix:", tag_matrix.shape)
print("Genome matrix:", genome_matrix.shape)
print("Weighted matrix:", weighted_matrix.shape)


In [5]:
import re


def normalize_title(title):
    title = str(title).strip().lower()

    # Remove year, e.g. "(1999)"
    title = re.sub(r"\s*\(\d{4}\)\s*$", "", title)

    # Convert "Matrix, The" → "The Matrix"
    if title.endswith(", the"):
        title = "the " + title[:-5]

    elif title.endswith(", a"):
        title = "a " + title[:-3]

    elif title.endswith(", an"):
        title = "an " + title[:-4]

    return title.strip()


movies["normalized_title"] = (
    movies["title"]
    .fillna("")
    .apply(normalize_title)
)


def recommend_movies(movie_title, n=10, min_rating_count=100):

    search_title = normalize_title(movie_title)

    # Exact normalized-title match.
    matches = movies[
        movies["normalized_title"] == search_title
    ]

    # Fallback to partial matching.
    if matches.empty:
        matches = movies[
            movies["normalized_title"].str.contains(
                search_title,
                regex=False,
                na=False
            )
        ]

    if matches.empty:
        print(f"Movie not found: {movie_title}")
        return pd.DataFrame()

    # DataFrame index matches the feature-matrix row.
    movie_index = matches.index[0]

    print(
        f"Selected movie: {movies.loc[movie_index, 'title']}"
    )

    # Use the weighted feature matrix.
    similarity_scores = cosine_similarity(
        weighted_matrix[movie_index],
        weighted_matrix
    ).flatten()

    candidates = movies.copy()
    candidates["similarity"] = similarity_scores

    # Remove the selected movie.
    candidates = candidates[
        candidates.index != movie_index
    ]

    # Remove movies with too few ratings.
    candidates = candidates[
        candidates["rating_count"] >= min_rating_count
    ]

    # Highest similarity first.
    candidates = candidates.sort_values(
        "similarity",
        ascending=False
    )

    return candidates[
        [
            "movieId",
            "clean_title",
            "year",
            "genres",
            "average_rating",
            "rating_count",
            "similarity"
        ]
    ].head(n).reset_index(drop=True)

In [ ]:
display(recommend_movies("Toy Story", 5))
display(recommend_movies("The Matrix", 5))
display(recommend_movies("The Godfather", 5))


In [ ]:
# ============================================================
#  QUALITATIVE EVALUATION
# ============================================================

test_movies = [
    "Toy Story",
    "The Matrix",
    "The Godfather",
    "Finding Nemo",
    "Titanic",
    "The Dark Knight",
    "Jumanji",
    "Shrek"
]

for movie in test_movies:

    print("\n" + "=" * 60)
    print(movie)
    print("=" * 60)

    result = recommend_movies(movie, 5)

    if result.empty:
        print("No recommendations found.")
    else:
        print(
            result[
                ["clean_title", "similarity"]
            ].to_string(index=False)
        )


In [19]:
from scipy.sparse import save_npz

save_npz(
    PROCESSED_DIR / "weighted_matrix.npz",
    weighted_matrix
)

movies.to_csv(
    PROCESSED_DIR / "phase2_movies.csv",
    index=False
)

print("Phase 2 artifacts saved.")

Phase 2 artifacts saved.
